# Funciones para Ejecución Modelo IA

### Función GPT

In [ ]:
def gpt_5_mini(client, text, system_instructions_used, response_format_used, initial_messages_used): 
    messages = copy.deepcopy(initial_messages_used)
    messages.insert(0, {"role": "system", "content": system_instructions_used})
    messages.append({"role": "user", "content": text})

    completion = client.beta.chat.completions.parse(
        model="gpt-5-mini",
        response_format = response_format_used,
        messages = messages,
    )
    response_content = completion.choices[0].message.content
    response_dict = json.loads(response_content)
    return response_dict

### Funciones Auxiliares de Configuración

In [ ]:
def debe_ejecutar_fase(fase_num, tipo_transaccion):
    """Determina si una fase debe ejecutarse según el tipo de transacción."""
    if fase_num == 5 and tipo_transaccion in df_config[df_config.omitir_fase_5 == True].tipo_transaccion.unique():
        return False
    return True

def obtener_modelo_dinamico(fase_num, contexto):
    fase_num = int(fase_num)
    tipo_transaccion = contexto.get('tipo_transaccion')
    numero_derechos = contexto.get('numero_derechos')
    
    # Obtener naturaleza. Asegurar que sea lista para la nueva función
    raw_naturaleza = contexto.get('naturaleza_agua')
    
    # Normalización simple: si viene un string, lo hacemos lista. Si es None, queda None.
    if isinstance(raw_naturaleza, str):
        lista_naturaleza = [raw_naturaleza]
    else:
        lista_naturaleza = raw_naturaleza # Asumimos que es list o None

    if fase_num == 3:
        return crear_modelo_fase3(tipo_transaccion)
    
    elif fase_num == 4:
        # Pasamos la lista
        return crear_modelo_fase4(numero_derechos, lista_naturaleza)
    
    elif fase_num == 7:
        return crear_modelo_fase7(tipo_transaccion)
    
    else:
        return globals()[f"Fase{fase_num}Registro"]

def construir_texto_con_contexto(texto_base, fase_num, contexto):
    texto_final = texto_base
    fase_num = int(fase_num)
    
    # Fases 3, 5, 7: agregar metadatos de clasificación
    if fase_num in [3, 5, 7] and 'tipo_transaccion' in contexto:
        tipo_trans = contexto['tipo_transaccion']
        
        try:
            # Buscamos la fila correspondiente en el df_config
            fila_config = df_config.loc[df_config['tipo_transaccion'] == tipo_trans]
            
            if not fila_config.empty:
                sub_rol = fila_config['sub_roles_fase_3'].values[0]
                rol = fila_config['roles_fase_3'].values[0]
                
                # Construimos el encabezado para guiar al modelo
                encabezado = (
                    f"### METADATOS DEL ACTO ###\n"
                    f"TIPO DE TRANSACCIÓN: {tipo_trans}\n"
                    f"CLASIFICACIÓN: {rol}\n"
                    f"SUBCLASIFICACIÓN: {sub_rol}\n"
                    f"###########################\n\n"
                )
                texto_final = f"{encabezado}TEXTO DEL DOCUMENTO A PROCESAR:\n\n{texto_final}"
            else:
                # Fallback si no está en df_config
                texto_final = f"TIPO DE TRANSACCIÓN: {tipo_trans}\n\nTEXTO DEL DOCUMENTO: \n\n{texto_final}"
        
        except Exception as e:
            print(f"Error al buscar clasificación: {e}")
            texto_final = f"TIPO DE TRANSACCIÓN: {tipo_trans}\n\nTEXTO DEL DOCUMENTO: \n\n{texto_final}"

    # Fase 4: agregar número de derechos
    elif fase_num == 4 and 'numero_derechos' in contexto:
        texto_final =   (
            f"CANTIDAD DE DERECHOS ESPERADA: {contexto['numero_derechos']}\n\n"
            f"TIPO DE DERECHO: {contexto['tipo_derecho']}\n\n"
            f"NATURALEZA DEL AGUA: {contexto['naturaleza_agua']}\n\n"
            f"EJERCICIO DEL DERECHO: {contexto['ejercicio_derecho']}\n\n"
            f"USO DEL AGUA: {contexto['uso_agua']}"
            f"###########################\n\n"
            f"\n\nTEXTO DEL DOCUMENTO: \n\n{texto_final}"
            )
    # Fase 6: agregar Conservador 
    elif fase_num == 6 and 'conservador_id' in contexto:
        texto_final = (
        f"CONSERVADOR DE BIENES RAÍCES: {contexto.get('conservador_id')}\n\n"
        f"INSCRIPCIÓN ACTUAL: {contexto.get('numero_cbr_actual')}, "
        f"{contexto.get('fojas_cbr_actual')}, {contexto.get('año_cbr_actual')}\n"
        f"###########################\n\n"
        f"TEXTO DEL DOCUMENTO: \n\n{texto_final}"
            )
    
    return texto_final

### Función de Creación de Modelos Dinámicos

In [ ]:
def crear_modelo_fase3(tipo_transaccion):
    """Crea modelo Fase3 con tipo_persona adaptado."""
    if tipo_transaccion in df_config[df_config.modificar_fase_3 == True].tipo_transaccion.unique():
        class Fase3RegistroModificado(BaseModel):
            tipo_persona: List[Emisor] = Field(description=descr["tipo_persona"])
            rut: List[RutType] = Field(description=descr["rut"])
            nombre: List[str] = Field(description=descr["nombre"])
            nombre_representante_legal: Optional[List[str]] = Field(None, description=descr["nombre_representante_legal"])
            rut_representante_legal: Optional[List[RutType]] = Field(None, description=descr["rut_representante_legal"])
            direccion: Optional[List[str]] = Field(None, description=descr["direccion"])
        
        return Fase3RegistroModificado
    
    # Modelo original
    return Fase3Registro


def crear_modelo_fase7(tipo_transaccion):
    """Crea modelo Fase7 adaptado según tipo de transacción."""
    if tipo_transaccion in df_config[df_config.modificar_fase_7 == True].tipo_transaccion.unique():
        class Fase7RegistroModificado(BaseModel):
            tipo_punto: List[CambioPuntoCaptacion] = Field(min_items=2, description=descr["tipo_punto"])
            nombre_punto: Optional[List[str]] = Field(None, min_items=2, description=descr["nombre_punto"])
            norte: Optional[List[float]] = Field(None, min_items=2, description=descr["norte"])
            este: Optional[List[float]] = Field(None,min_items=2, description=descr["este"])
            datum: Optional[Datum] = Field(None, description=descr["datum"])
            huso: Optional[Huso] = Field(None, description=descr["huso"])
        
        return Fase7RegistroModificado
    
    return Fase7Registro

In [ ]:
from enum import Enum
from typing import List, Optional, Annotated, Dict, Any
from pydantic import BaseModel, Field

# --- DICCIONARIOS DE FUENTES ADMISIBLES ---
# Clave: Nombre del atributo en el Enum (código)
# Valor: Valor real del Enum (lo que se ve en el JSON)

FUENTES_SUBTERRANEA = {
    "POZO": "POZO (PROFUNDO)",
    "NORIA": "NORIA",
    "PUNTERA": "PUNTERA",
    "DREN": "DREN",
    "GALERIA": "GALERÍA / SOCAVÓN"
}

FUENTES_CORRIENTE = {
    "RIO_ESTERO": "RÍO / ESTERO",
    "CANAL": "CANAL",
    "VERTIENTE": "VERTIENTE",
    "ARROYO": "ARROYO",
    "CHORRILLO": "CHORRILLO",
    "ZANJON": "ZANJÓN",
    "QUEBRADA": "QUEBRADA"
}

FUENTES_DETENIDA = {
    "CHARCA_AGUADA": "CHARCA / AGUADA",
    "DERRAME": "DERRAME",
    "EMBALSE_TRANQUE": "EMBALSE / TRANQUE",
    "LAGO_LAGUNA": "LAGO / LAGUNA",
    "ZANJON": "ZANJÓN",  # Zanjón aparece en ambos en tu definición previa
    "VERTIENTE": "VERTIENTE" # Vertiente a veces se considera detenida si es captación en la fuente misma
}

# Unimos todo para el caso de fallback (cuando no hay info o es "NO INDICA")
FUENTES_TODAS = {**FUENTES_SUBTERRANEA, **FUENTES_CORRIENTE, **FUENTES_DETENIDA, "OTRO": "OTRO"}

In [ ]:
def crear_modelo_fase4(numero_derechos: int, naturaleza_agua: List[str] = None):
    """
    Crea modelo Fase4 dinámico.
    naturaleza_agua: Debe ser una lista, ej: ["SUBTERRÁNEA", "SUPERFICIAL Y DETENIDA"]
    """

    # 1. Construcción dinámica del diccionario de fuentes permitidas
    fuentes_permitidas = {}
    
    # Si la lista es None o está vacía, o contiene "NO INDICA", habilitamos TODO.
    if not naturaleza_agua or any(x in ["NO INDICA", "NO INDICA/OTRO"] for x in naturaleza_agua):
        fuentes_permitidas = FUENTES_TODAS
    else:
        # Iteramos sobre cada naturaleza detectada y sumamos sus fuentes al diccionario
        for nat in naturaleza_agua:
            if nat == "SUBTERRÁNEA":
                fuentes_permitidas.update(FUENTES_SUBTERRANEA)
            elif nat == "SUPERFICIAL Y CORRIENTE" or nat == "SUPERFICIAL CORRIENTES/DETENIDAS":
                 # Nota: Ajusta los strings según tus datos exactos de Fase 2
                fuentes_permitidas.update(FUENTES_CORRIENTE)
            elif nat == "SUPERFICIAL Y DETENIDA":
                fuentes_permitidas.update(FUENTES_DETENIDA)
    
    # Si por alguna razón la lista tenía valores raros y el dict quedó vacío, usamos fallback
    if not fuentes_permitidas:
        fuentes_permitidas = FUENTES_TODAS

    # 2. Creación del Enum Dinámico
    # Sintaxis: Enum(NombreClase, DiccionarioDeMiembros, type=str)
    # Esto crea una clase equivalente a class TipoFuenteDinamico(str, Enum): ...
    TipoFuenteDinamico = Enum('TipoFuenteDinamico', fuentes_permitidas, type=str)

    # Si numero_derechos es None (fallback estático), devolvemos la clase base genérica
    # OJO: Para consistencia, aquí podrías devolver la dinámica con min_items=0
    if numero_derechos is None:
        return Fase4Registro 

    # 3. Construcción del Modelo Pydantic
    class Fase4RegistroDinamico(BaseModel):
        tipo_caudal: List[TipoCaudal] = Field(
            min_items=numero_derechos, 
            max_items=numero_derechos, 
            description=descr["tipo_caudal"]
        )
        unidad_caudal: UnidadCaudal = Field(description=descr["unidad_caudal"])
        
        caudal_promedio: Optional[List[PositiveFloat]] = Field(
            None, 
            min_items=numero_derechos, 
            max_items=numero_derechos, 
            description=descr["caudal_promedio"]
        )
        
        # --- Campo con Enum Dinámico ---
        tipo_fuente: Optional[List[TipoFuenteDinamico]] = Field(
            None, 
            min_items=numero_derechos, 
            max_items=numero_derechos, 
            description=f"Tipo de fuente restringido a las naturalezas: {naturaleza_agua}"
        )
        
        fuente: Optional[List[str]] = Field(
            None, 
            min_items=numero_derechos, 
            max_items=numero_derechos, 
            description=descr["fuente"]
        )
        cuenca: Optional[List[str]] = Field(
            None, 
            min_items=numero_derechos, 
            max_items=numero_derechos, 
            description=descr["cuenca"]
        )

    # Agregar campos mensuales (lógica existente)
    for mes in ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']:
        campo = f'caudal_{mes}'
        Fase4RegistroDinamico.__annotations__[campo] = Optional[List[Optional[PositiveFloat]]]
        setattr(
            Fase4RegistroDinamico,
            campo,
            Field(None, description=f"Caudal para el mes de {mes}.")
        )
    
    Fase4RegistroDinamico.model_rebuild()
    return Fase4RegistroDinamico

In [ ]:
def obtener_instr_dinamica(fase_num, contexto, fase_config):
    
    if int(fase_num) == 3:
        # Extraemos el tipo detectado en Fase 1 o 2 desde el contexto
        tipo_trans = contexto.get("tipo_transaccion")
        
        # 2. Buscar el sub-rol con manejo de errores
        try:
            sub_rol = df_config.loc[df_config['tipo_transaccion'] == tipo_trans, 'sub_roles_fase_3'].values[0]
        except (IndexError, KeyError):
            sub_rol = "NO INDICA/ERROR"
        
        # Obtenemos el texto específico de naturaleza
        naturaleza_texto = diccionario_naturaleza.get(sub_rol)
        fase_instr_especifica = instr_fase3.replace("[[naturaleza]]", naturaleza_texto)
        system_instr = instr_base + "\n\n" + fase_instr_especifica
    
    else:
        # Lógica estándar para el resto de las fases
        system_instr = instr_base + "\n\n" + fase_config["instr"]
    
    return system_instr

### Función Principal de Procesamiento de la Fase

In [ ]:
def procesar_fase_gpt(client, texto_base, contexto, fase_config, fase_num):
    """Procesa una fase con el modelo GPT Correspondiente."""
    # Construir texto con contexto
    texto_final = construir_texto_con_contexto(texto_base, fase_num, contexto)

    # Instrucciones del sistema
    system_instr = obtener_instr_dinamica(fase_num, contexto, fase_config)
    
    # Obtener modelo dinámico
    response_model = obtener_modelo_dinamico(fase_num, contexto)
    
    # Preparar ejemplos
    initial_messages = []
    for ex in fase_config["examples"]:
        initial_messages.append({"role": "user", "content": ex["input"]})
        initial_messages.append({"role": "assistant", "content": json.dumps(ex["output"])})
    
    # Llamar al modelo
    response = gpt_5_mini(client, texto_final, system_instr, response_model, initial_messages)
    
    return response